In [1]:
!pip install langchain chromadb faiss-cpu langchain-google-genai google-genai langchain-community wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [1]:
from langchain_community.retrievers import WikipediaRetriever

retriever=WikipediaRetriever(top_k_results=2 , lang="en")

query="the geographical history of pakistan and india from the perspective of chines"

docs=retriever.invoke(query)


/tmp/ipykernel_7163/817736677.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [11]:
# wikipedia retriever
import wikipedia
from langchain_community.retrievers import WikipediaRetriever

# Configure the wikipedia library to use a custom User-Agent header
wikipedia.set_user_agent("MyLangChainApp/1.0 (contact: your_email@example.com)")

# Re-initialize retriever
retriever = WikipediaRetriever(top_k_results=5, lang="en")

# Let's try executing your original query
query = "The history about cricket"
try:
    docs = retriever.invoke(query)
    print("Successfully retrieved documents!")
    for i, doc in enumerate(docs):
        print(f"\nDocument {i+1}:")
        print(f"Title: {doc.metadata.get('title')}")
        print(f"Source: {doc.metadata.get('source')}")
        print(f"Content snippet: {doc.page_content[:300]}...")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved documents!

Document 1:
Title: History of cricket
Source: https://en.wikipedia.org/wiki/History_of_cricket
Content snippet: The sport of cricket has a known history beginning in the late 16th century England. It became an established sport in the country in the 18th century and developed globally in the 19th and 20th centuries. International matches have been played since the 19th-century and formal Test cricket matches ...

Document 2:
Title: Poetry about cricket
Source: https://en.wikipedia.org/wiki/Poetry_about_cricket
Content snippet: The game of cricket has inspired much poetry, most of which romanticises the sport and its culture.


== Poems ==
The first known poem about cricket was written in Latin hexameters, by the Bristol schoolmaster William Goldwyn in 1704. Notable authors and poets who created poetry about cricket includ...

Document 3:
Title: History of cricket to 1725
Source: https://en.wikipedia.org/wiki/History_of_cricket_to_1725
Content snippet:

In [10]:

# Vector Store Retriever
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata
import os

# Securely load your Gemini API Key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

from langchain_core.documents import Document
#source documents
documents=[
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose"),
    Document(page_content="A bunch of scientists bring back an extra ordinary exepriment that can help people"),
    Document(page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ..."),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and "),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them"),
]

embedding_model=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
# creating chroma vectore store in memory
vectore_store=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)
# convert a vector store into retrievers
retriever=vectore_store.as_retriever(search_kwargs=dict(k=2))
# we can make use of multiple search strageties using retrievers instead using vectorestore.similarity_search that can only search based on a single strategy used by default
query="what do you know about scientists"
results=retriever.invoke(query)
print(results)




[Document(metadata={}, page_content='A bunch of scientists bring back an extra ordinary exepriment that can help people'), Document(metadata={}, page_content='A bunch of scientists bring back an extra ordinary exepriment that can help people')]


In [ ]:
# maximum marginal relevance (MMR)
